In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import sys
from pathlib import Path

repo_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "src" / "training" / "data_utils.py").is_file()
)
sys.path.insert(0, str(repo_root / "src"))
from training.data_utils import canonical_legacy_track_id_split



In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
from sklearn.neural_network import MLPRegressor

In [ ]:
from sklearn.svm import SVR

In [ ]:
from catboost import CatBoostRegressor

In [ ]:
df1 = pd.read_csv(repo_root / "data" / "SpotGenTrack" / "Data Sources" / "spotify_tracks.csv")
df2 = pd.read_csv(repo_root / "data" / "SpotGenTrack" / "Features Extracted" / "lyrics_features.csv")

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
csv1.head()

In [ ]:
csv2.head()

In [ ]:
df2_clean = df2[df2['mean_syllables_word'] != -1].copy()

df1_popularity = df1[['id', 'popularity']].copy()

df = pd.merge(
    df2_clean, 
    df1_popularity, 
    left_on='track_id', 
    right_on='id', 
    how='inner'
)

df = df.drop(columns=['id'])

columns_to_drop = [col for col in df.columns if 'Unnamed' in col]
df = df.drop(columns=columns_to_drop)

In [ ]:
df.head()

In [ ]:
feature_cols = [
    'mean_syllables_word',
    'mean_words_sentence',
    'n_sentences',
    'n_words',
    'sentence_similarity',
    'vocabulary_wealth'
]

clean_df = df.dropna(subset=feature_cols + ['popularity', 'track_id']).copy()
track_ids = clean_df['track_id'].astype(str)
X = clean_df[feature_cols]
y = clean_df['popularity']

train_ids, val_ids, test_ids = map(
    set, canonical_legacy_track_id_split()
)
train_mask = track_ids.isin(train_ids)
val_mask = track_ids.isin(val_ids)
test_mask = track_ids.isin(test_ids)

X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_val, y_val = X.loc[val_mask], y.loc[val_mask]
X_test, y_test = X.loc[test_mask], y.loc[test_mask]
print(f"Legacy split subset: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

rf_model = RandomForestRegressor(n_estimators=250, random_state=10)
rf_model.fit(X_train_scaled, y_train)

preds = rf_model.predict(X_test_scaled)
rmse = np.sqrt(mean_squared_error(y_test, preds))
mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)

print("=== Random Forest Performance ===")
print(f"rmse: {rmse:.4f}")
print(f"mae:    {mae:.4f}")
print(f"r2: {r2:.4f}\n")

importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=importance_df, x='Importance', y='Feature', palette='viridis')
plt.title('Lyrical Feature Importances for Popularity Prediction')
plt.xlabel('Importance Score')
plt.ylabel('Lyric Feature')
plt.tight_layout()
plt.show()


In [ ]:

rf_model = RandomForestRegressor(n_estimators=250, random_state=10)
rf_model.fit(X_train_scaled, y_train)

preds = rf_model.predict(X_test_scaled)
rmse = np.sqrt(mean_squared_error(y_test, preds))
mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)

print("=== Random Forest Performance ===")
print(f"rmse: {rmse:.4f}")
print(f"mae:    {mae:.4f}")
print(f"r2: {r2:.4f}\n")


importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=importance_df, x='Importance', y='Feature', palette='viridis')
plt.title('Lyrical Feature Importances for Popularity Prediction')
plt.xlabel('Importance Score')
plt.ylabel('Lyric Feature')
plt.tight_layout()
plt.show()


=== Random Forest Performance ===  (all features)
RMSE (Root Mean Squared Error): 13.2630
MAE  (Mean Absolute Error):    9.9282
R² Score:                       0.4111



=== gb Performance === (all features)
RMSE (Root Mean Squared Error): 15.2922
MAE  (Mean Absolute Error):    12.3210
R² Score:                       0.2171



In [ ]:
gb_model = GradientBoostingRegressor(
    n_estimators=500, 
    learning_rate=0.02, 
    max_depth=4, 
    random_state=42
)
gb_model.fit(X_train_scaled, y_train)

In [ ]:
preds = gb_model.predict(X_test_scaled)
rmse = np.sqrt(mean_squared_error(y_test, preds))
mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)

print("=== gb Performance ===")
print(f"RMSE (Root Mean Squared Error): {rmse:.4f}")
print(f"MAE  (Mean Absolute Error):    {mae:.4f}")
print(f"R² Score:                       {r2:.4f}\n")


importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': gb_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=importance_df, x='Importance', y='Feature', palette='viridis')
plt.title('Lyrical Feature Importances for Popularity Prediction')
plt.xlabel('Importance Score')
plt.ylabel('Lyric Feature')
plt.tight_layout()
plt.show()

In [ ]:
models = {
    "SVR (RBF)": SVR(C=10.0),
    "MLP (Neural Net)": MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
}

results = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    
    r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    
    results.append({"Model": name, "R2 Score": r2, "MAE": mae, "RMSE": rmse})

# Convert to DataFrame for quick visual comparison
results_df = pd.DataFrame(results).sort_values(by="R2 Score", ascending=False)
print(results_df)

In [ ]:
s